<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/gaussiennes_dir.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score
import time

# ── HELPERS classification ────────────────────────────────────────────────
def cls_acc(f, y):
    sg = np.sign(f)
    return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
def cls_auc(f, y):
    try: return float(roc_auc_score((y > 0).astype(int), f))
    except Exception: return float('nan')
def cls_str(f, y):
    return f"acc={cls_acc(f,y):.4f} AUC={cls_auc(f,y):.4f}"

params = {
    "n_ambiant": 5, "deg_P": 3, "n_terms_poly": 20,
    "seeds": {1: 599453, 2: 25954, 3: 56605,
              42: 5905412, 43: 6852036, 44: 4152403, 11: 228244, 12: 7492241},
    "weights": {0: 0, 1: 1, 2: 0.1, 3: 0.01},

    "n_train": 50, "n_test": 5000,

    "n_experts": 6,             # nb de niveaux gaussiens anisotropes INDÉPENDANTS
    "n_dict": 2000, "n_centres": 400,
    "sigma_min": 0.1, "sigma_max": 3.0,
    "lambda_reg": 1e-5,    "thresh_factor": 1,

    "n_G": 2000,                 # points cloud pour le Gram Sobolev de Phase 1
    "n_G_H": 3000,                # points cloud pour le Gram Sobolev de Phase 2
    "k_loss": 3.0,               # sélection Phase 2 : garde loss <= k_loss * meilleure
    "n_dirs": 50, "batch_dirs": 10,   # directionnel, par blocs (cf version jointe)
}
params["n_unlabeled"] = np.maximum(4000 - params["n_train"], 500)
params["train_center_ratio"] = 0.5 + params["n_train"] / (2 * 4000)

# ══════════════════════════════════════════════════════════════════════════════
# POLYNÔMES ET CLOUD SUR {Q=0}  (repris tel quel, code déjà validé)
# ══════════════════════════════════════════════════════════════════════════════
def random_sparse_polynomial(d, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [exp for exp in product(range(degree+1), repeat=d) if sum(exp) <= degree]
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            term = np.ones(x.shape[0])
            for j, e in enumerate(alpha):
                if e > 0: term *= x[:, j]**e
            y += c * term
        return y
    zero_val = sum(c for c, alpha in zip(coeffs, indices) if all(e == 0 for e in alpha))
    def P_zero(x): return P(x) - zero_val
    return P_zero, indices, coeffs

def normalize_polynomial(P, indices, coeffs):
    max_c = np.max(np.abs(coeffs))
    if max_c > 0:
        nc = coeffs / max_c
        def Pn(x):
            y = np.zeros(x.shape[0])
            for c, alpha in zip(nc, indices):
                term = np.ones(x.shape[0])
                for j, e in enumerate(alpha):
                    if e > 0: term *= x[:, j]**e
                y += c * term
            return y
        return Pn
    return P

P1, i1, c1 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][1])
P2, i2, c2 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][2])
P3, i3, c3 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][3])
P1 = normalize_polynomial(P1, i1, c1); P2 = normalize_polynomial(P2, i2, c2); P3 = normalize_polynomial(P3, i3, c3)
def Q(x): return P1(x)*P2(x)*P3(x)

def grad_Q_analytical(X):
    eps = 1e-5; d = X.shape[1]; p1 = P1(X); p2 = P2(X); p3 = P3(X)
    grad = np.zeros_like(X)
    for k in range(d):
        Xp = X.copy(); Xp[:, k] += eps; Xm = X.copy(); Xm[:, k] -= eps
        dp1 = (P1(Xp)-P1(Xm))/(2*eps); dp2 = (P2(Xp)-P2(Xm))/(2*eps); dp3 = (P3(Xp)-P3(Xm))/(2*eps)
        grad[:, k] = dp1*p2*p3 + p1*dp2*p3 + p1*p2*dp3
    return grad

def project_to_Q_zero(X_init, n_steps=60, tol=1e-4, damp=1.0):
    X = X_init.copy()
    for _ in range(n_steps):
        q = Q(X); gq = grad_Q_analytical(X)
        g2 = np.sum(gq*gq, axis=1, keepdims=True) + 1e-12
        X = X - damp*(q[:, None]*gq)/g2
        X = np.clip(X, 0., 1.)
        if np.abs(Q(X)).max() < tol*0.1:
            break
    return X, np.abs(Q(X)) < tol

def sample_on_Q_zero(n_target, d, seed=None, max_batches=200):
    rng = np.random.default_rng(seed)
    collected = []; n_col = 0; n_seen = 0; n_ok = 0
    for _ in range(max_batches):
        if n_col >= n_target:
            break
        n_batch = min(max((n_target - n_col)*4, 200), 20000)
        Xi = rng.uniform(0, 1, (int(n_batch), d))
        Xp, conv = project_to_Q_zero(Xi)
        good = Xp[conv]
        good = good[np.all(np.isfinite(good), axis=1)]
        n_seen += len(Xi); n_ok += int(conv.sum())
        if len(good) > 0:
            collected.append(good); n_col += len(good)
        print(f"  collectés:{n_col}/{n_target} (cumulé {n_ok/max(n_seen,1):.1%})", end='\r')
    print()
    if n_col < n_target:
        raise RuntimeError(f"sample_on_Q_zero: seulement {n_col}/{n_target} points.")
    return np.vstack(collected)[:n_target]

print("Génération du cloud sur {Q=0}...")
d = params["n_ambiant"]
X_train = sample_on_Q_zero(params["n_train"], d, seed=params["seeds"][42])
X_test = sample_on_Q_zero(params["n_test"], d, seed=params["seeds"][43])
X_unlabeled = sample_on_Q_zero(params["n_unlabeled"], d, seed=params["seeds"][44])
print(f"  X_train:{X_train.shape}  X_unlabeled:{X_unlabeled.shape}")

P_target1, indicest1, coeffst1 = random_sparse_polynomial(params["n_ambiant"], 4, params["n_terms_poly"], seed=params["seeds"][11])
P_target2, _, _ = random_sparse_polynomial(params["n_ambiant"], 4, params["n_terms_poly"], seed=params["seeds"][12])
Ptarget1 = normalize_polynomial(P_target1, indicest1, coeffst1)
Ptarget2 = normalize_polynomial(P_target2, indicest1, coeffst1)

def target_function(X):
    return np.minimum(np.abs(P_target1(X)), 1) + np.minimum(np.abs(P_target2(X)), 1)

y_cont_train = target_function(X_train); y_cont_test = target_function(X_test)
X_all = np.vstack([X_train, X_unlabeled]); N_all = len(X_all)

_med = float(np.median(target_function(X_unlabeled)))
y_train = np.where(y_cont_train >= _med, 1.0, -1.0)
y_test = np.where(y_cont_test >= _med, 1.0, -1.0)
print(f"  cible binarisée au seuil médian={_med:.4f} : "
      f"train (+1)={int((y_train>0).sum())}/{len(y_train)}  "
      f"test (+1)={int((y_test>0).sum())}/{len(y_test)}")
_bal_te = (y_test > 0).mean()
if _bal_te < 0.3 or _bal_te > 0.7:
    print(f"  [!] classes déséquilibrées ({_bal_te:.1%} de +1) — préférer l'AUC.")


# ══════════════════════════════════════════════════════════════════════════════
# GAUSSIENNES ANISOTROPES DIRECTIONNELLES — un sigma PAR DIMENSION par centre.
# Convention phi(x)=exp(-Σ(x_i-c_i)²/(2σ_i²)). Directionnelle : le long d'une
# droite, une gaussienne (même anisotrope) reste gaussienne en t — formule
# fermée valable, juste pondérée par 1/σ_i² par dimension au lieu d'un scalaire.
# Mêmes constantes de calibration qu'en isotrope (d, d(d+2), d(d+2)(d+4), 9) —
# validées numériquement contre le calcul dense anisotrope (~3-5% de bruit MC).
# Par BLOCS de directions (comme la version jointe) — mémoire de crête
# indépendante de n_dirs.
# ══════════════════════════════════════════════════════════════════════════════
def gaussian_features_aniso(X, centers, sigmas):
    diff = X[:, None, :] - centers[None, :, :]
    sq = np.sum(diff**2/sigmas[None, :, :]**2, axis=2)
    return np.exp(-sq/2)

def sample_candidates_aniso(X_train, X_unlabeled, n_candidates, train_ratio, rng, d):
    n_tr = min(int(round(n_candidates*train_ratio)), X_train.shape[0])
    n_ul = min(n_candidates-n_tr, X_unlabeled.shape[0]); parts = []
    if n_tr > 0: parts.append(X_train[rng.choice(X_train.shape[0], n_tr, replace=False)])
    if n_ul > 0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0], n_ul, replace=False)])
    centers = np.vstack(parts)
    log_s = rng.uniform(np.log(params["sigma_min"]), np.log(params["sigma_max"]),
                        size=(len(centers), d))
    sigmas = np.exp(log_s)
    return centers, sigmas

def build_G_aniso_directional_streaming(X_cloud, centers, sigmas, weights, n_dirs,
                                        batch_dirs=10, rng=None):
    """Gram Sobolev d'un niveau (gaussiennes anisotropes), par blocs de
    directions — jamais n_dirs directions en mémoire simultanément."""
    if rng is None:
        rng = np.random.default_rng(0)
    n, d_ = X_cloud.shape
    m = centers.shape[0]
    diff = X_cloud[:, None, :] - centers[None, :, :]        # (n,m,d)
    inv2 = 1.0/sigmas**2                                    # (m,d)
    sq = np.sum(diff**2*inv2[None, :, :], axis=2)           # (n,m)
    phi = np.exp(-sq/2)

    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.)
    w2 = weights.get(2, 0.); w3 = weights.get(3, 0.)
    need2 = w2 != 0; need3 = w3 != 0

    G = np.zeros((m, m))
    if w0:
        G += w0*(phi.T@phi)/n

    if need2:
        # Laplacien anisotrope exact — indépendant des directions
        TR = (-np.sum(inv2, axis=1)[None, :] + np.sum(diff**2*inv2[None, :, :]**2, axis=2))*phi
        TRG = (TR.T@TR)/n
    if need3:
        # grad(Laplacien) anisotrope exact, O(n·m·d)
        Csum = -np.sum(inv2, axis=1)                        # (m,)
        Dq = np.sum(diff**2*inv2[None, :, :]**2, axis=2)     # (n,m)
        coefV = 2*inv2[None, :, :]**2 - (Csum[None, :, None]+Dq[:, :, None])*inv2[None, :, :]
        V = phi[:, :, None]*diff*coefV                        # (n,m,d)
        VVG = np.einsum('nid,njd->ij', V, V)/n

    MC1_sum = np.zeros((m, m)); MC2_sum = np.zeros((m, m)); MC3_sum = np.zeros((m, m))
    done = 0
    while done < n_dirs:
        K = min(batch_dirs, n_dirs - done)
        U = rng.normal(size=(n, K, d_))
        U /= np.linalg.norm(U, axis=2, keepdims=True)
        # A[n,m,k] = Σ_d diff[n,m,d]·U[n,k,d]/σ_m,d²   ;   B[n,m,k] = Σ_d U[n,k,d]²/σ_m,d²
        A = np.einsum('nmd,nkd,md->nmk', diff, U, inv2)
        B = np.einsum('nkd,md->nmk', U**2, inv2)
        gp = -A; gpp = -B
        if w1:
            D1 = gp*phi[:, :, None]
            D1f = D1.transpose(0, 2, 1).reshape(-1, m)
            MC1_sum += D1f.T@D1f
            del D1, D1f
        if need2 or need3:
            D2 = (gpp+gp**2)*phi[:, :, None]
            if need2:
                D2f = D2.transpose(0, 2, 1).reshape(-1, m)
                MC2_sum += D2f.T@D2f
                del D2f
            if need3:
                D3 = (3*gp*gpp+gp**3)*phi[:, :, None]
                D3f = D3.transpose(0, 2, 1).reshape(-1, m)
                MC3_sum += D3f.T@D3f
                del D3, D3f
            del D2
        del U, A, B, gp, gpp
        done += K

    if w1:
        G += w1*d_*MC1_sum/(n*n_dirs)
    if need2:
        MC2 = MC2_sum/(n*n_dirs)
        G += w2*(d_*(d_+2)*MC2 - TRG)/2
    if need3:
        MC3 = MC3_sum/(n*n_dirs)
        G += w3*(d_*(d_+2)*(d_+4)*MC3 - 9*VVG)/6
    return G


# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 : n_experts niveaux gaussiens anisotropes INDÉPENDANTS (pas de glouton)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 1 : {params['n_experts']} experts gaussiens anisotropes indépendants...")

F_train = np.zeros((len(X_train), params["n_experts"]))
F_test = np.zeros((len(X_test), params["n_experts"]))
experts = []   # liste de dicts {'centers','sigmas','coeffs'}
times_p1 = []

for e in range(params["n_experts"]):
    t0 = time.time()
    rng = np.random.default_rng(seed=100+e)
    candidates, sigmas_cand = sample_candidates_aniso(
        X_train, X_unlabeled, params["n_dict"], params["train_center_ratio"], rng, d)

    A_cand = gaussian_features_aniso(X_train, candidates, sigmas_cand)
    corr = (A_cand.T@y_train)/len(X_train)
    scores = corr**2

    k = min(params["n_centres"], len(candidates))
    top_k = np.argsort(scores)[-k:]
    centers = candidates[top_k]; sigmas_sel = sigmas_cand[top_k]
    A = A_cand[:, top_k]
    Atest = gaussian_features_aniso(X_test, centers, sigmas_sel)

    n_G = min(N_all, params["n_G"])
    idx_G = rng.choice(N_all, size=n_G, replace=False)
    G = build_G_aniso_directional_streaming(
        X_all[idx_G], centers, sigmas_sel, params["weights"],
        n_dirs=params["n_dirs"], batch_dirs=params["batch_dirs"],
        rng=np.random.default_rng(9000+e))

    n = A.shape[0]
    M = (A.T@A)/n + params["lambda_reg"]*G
    rhs = (A.T@y_train)/n
    eigvals, eigvecs = np.linalg.eigh(M)
    thresh = max(params["lambda_reg"]*params["thresh_factor"], 1e-15); mask = eigvals > thresh
    V = eigvecs[:, mask]; S = eigvals[mask]; coeffs = V@((V.T@rhs)/S)

    f_tr = A@coeffs; f_te = Atest@coeffs
    loss_data = np.mean((y_train - f_tr)**2)
    loss_reg = params["lambda_reg"]*coeffs@G@coeffs
    t1 = time.time(); times_p1.append(t1-t0)

    F_train[:, e] = f_tr; F_test[:, e] = f_te
    experts.append({'centers': centers, 'sigmas': sigmas_sel, 'coeffs': coeffs})

    print(f"  expert {e+1}/{params['n_experts']} | rang={mask.sum():3d}/{k} | "
          f"loss={loss_data+loss_reg:.6f} (data={loss_data:.6f} reg={loss_reg:.6f}) | "
          f"{cls_str(f_te, y_test)} | {times_p1[-1]:.1f}s")


# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 : Gram H entre experts (fonctions AGRÉGÉES, directionnel partagé)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 2 : Gram H entre {params['n_experts']} experts (n_G_H={params['n_G_H']})...")

rng_H = np.random.default_rng(seed=999)
idx_H = rng_H.choice(N_all, size=min(N_all, params["n_G_H"]), replace=False)
X_H = X_all[idx_H]; n_H = len(idx_H)

w0 = params["weights"].get(0, 0.); w1 = params["weights"].get(1, 0.)
w2 = params["weights"].get(2, 0.); w3 = params["weights"].get(3, 0.)
need2 = w2 != 0; need3 = w3 != 0
n_exp = params["n_experts"]

# ── grandeurs agrégées par expert, indépendantes des directions ──
PHI_H = np.zeros((n_H, n_exp))
TR_H = np.zeros((n_H, n_exp)) if need2 else None
V_H = np.zeros((n_H, n_exp, d)) if need3 else None

for e, ex in enumerate(experts):
    diff = X_H[:, None, :] - ex['centers'][None, :, :]
    inv2 = 1.0/ex['sigmas']**2
    sq = np.sum(diff**2*inv2[None, :, :], axis=2)
    phi_k = np.exp(-sq/2)                                   # (n_H, m_e)
    PHI_H[:, e] = phi_k@ex['coeffs']
    if need2:
        TR_k = (-np.sum(inv2, axis=1)[None, :] + np.sum(diff**2*inv2[None, :, :]**2, axis=2))*phi_k
        TR_H[:, e] = TR_k@ex['coeffs']
    if need3:
        Csum = -np.sum(inv2, axis=1)
        Dq = np.sum(diff**2*inv2[None, :, :]**2, axis=2)
        coefV = 2*inv2[None, :, :]**2 - (Csum[None, :, None]+Dq[:, :, None])*inv2[None, :, :]
        V_k = phi_k[:, :, None]*diff*coefV                    # (n_H, m_e, d)
        V_H[:, e, :] = np.einsum('nmd,m->nd', V_k, ex['coeffs'])

G_H = np.zeros((n_exp, n_exp))
if w0:
    G_H += w0*(PHI_H.T@PHI_H)/n_H
if need2:
    TRG_H = (TR_H.T@TR_H)/n_H
if need3:
    VVG_H = np.einsum('nid,njd->ij', V_H, V_H)/n_H

# ── partie directionnelle, PARTAGÉE entre tous les experts, par blocs ──
if w1 or need2 or need3:
    rng_dir = np.random.default_rng(seed=8000)
    MC1_sum = np.zeros((n_exp, n_exp)); MC2_sum = np.zeros((n_exp, n_exp)); MC3_sum = np.zeros((n_exp, n_exp))
    done = 0
    while done < params["n_dirs"]:
        K = min(params["batch_dirs"], params["n_dirs"] - done)
        U = rng_dir.normal(size=(n_H, K, d))
        U /= np.linalg.norm(U, axis=2, keepdims=True)

        D1_agg = np.zeros((n_H, K, n_exp)) if w1 else None
        D2_agg = np.zeros((n_H, K, n_exp)) if (need2 or need3) else None
        D3_agg = np.zeros((n_H, K, n_exp)) if need3 else None

        for e, ex in enumerate(experts):
            diff = X_H[:, None, :] - ex['centers'][None, :, :]
            inv2 = 1.0/ex['sigmas']**2
            sq = np.sum(diff**2*inv2[None, :, :], axis=2)
            phi_k = np.exp(-sq/2)
            A = np.einsum('nmd,nkd,md->nmk', diff, U, inv2)
            B = np.einsum('nkd,md->nmk', U**2, inv2)
            gp = -A; gpp = -B
            if w1:
                D1_k = gp*phi_k[:, :, None]
                D1_agg[:, :, e] = np.einsum('nmk,m->nk', D1_k, ex['coeffs'])
            if need2 or need3:
                D2_k = (gpp+gp**2)*phi_k[:, :, None]
                D2_agg[:, :, e] = np.einsum('nmk,m->nk', D2_k, ex['coeffs'])
            if need3:
                D3_k = (3*gp*gpp+gp**3)*phi_k[:, :, None]
                D3_agg[:, :, e] = np.einsum('nmk,m->nk', D3_k, ex['coeffs'])

        if w1:
            D1f = D1_agg.reshape(-1, n_exp)
            MC1_sum += D1f.T@D1f
        if need2:
            D2f = D2_agg.reshape(-1, n_exp)
            MC2_sum += D2f.T@D2f
        if need3:
            D3f = D3_agg.reshape(-1, n_exp)
            MC3_sum += D3f.T@D3f
        done += K

    if w1:
        G_H += w1*d*MC1_sum/(n_H*params["n_dirs"])
    if need2:
        MC2 = MC2_sum/(n_H*params["n_dirs"])
        G_H += w2*(d*(d+2)*MC2 - TRG_H)/2
    if need3:
        MC3 = MC3_sum/(n_H*params["n_dirs"])
        G_H += w3*(d*(d+2)*(d+4)*MC3 - 9*VVG_H)/6

print(f"  G_H calculée ({n_H} pts, {params['n_dirs']} directions partagées)")

# ── sélection des experts par loss (k_loss) ──
n_loc = len(y_train)
m_vec = F_train.T@y_train/n_loc
q_vec = np.sum(F_train**2, axis=0)/n_loc
denom = q_vec + params["lambda_reg"]*np.diag(G_H)
bad_denom = denom <= 1e-14
denom_safe = np.where(bad_denom, 1.0, denom)
expert_losses = np.mean(y_train**2) - m_vec**2/denom_safe
expert_losses = np.where(bad_denom, np.inf, expert_losses)

best_loss = expert_losses.min()
sel = expert_losses <= params["k_loss"]*best_loss
sel_idx = np.where(sel)[0]
print(f"\n  Sélection experts (k_loss={params['k_loss']}) : {sel.sum()}/{n_exp} retenus")
for i in range(n_exp):
    tag = "GARDÉ " if sel[i] else "écarté"
    print(f"    expert{i+1} : loss={expert_losses[i]:.3e}  [{tag}]")

F_train_sel = F_train[:, sel]; F_test_sel = F_test[:, sel]
G_H_sel = G_H[np.ix_(sel_idx, sel_idx)]

n_tr = F_train_sel.shape[0]
M_H = (F_train_sel.T@F_train_sel)/n_tr + params["lambda_reg"]*G_H_sel
rhs_H = (F_train_sel.T@y_train)/n_tr
eigvals_H, eigvecs_H = np.linalg.eigh(M_H)
thresh_H = max(params["lambda_reg"]*params["thresh_factor"], 1e-15)
mask_H = eigvals_H > thresh_H
V_Hm = eigvecs_H[:, mask_H]; S_H = eigvals_H[mask_H]
alpha_sel = V_Hm@((V_Hm.T@rhs_H)/S_H)

pred_H_train = F_train_sel@alpha_sel
pred_H_test = F_test_sel@alpha_sel

print(f"\n  rang H = {mask_H.sum()}/{sel.sum()} (sur {n_exp} experts)")
print(f"  train : {cls_str(pred_H_train, y_train)}   test : {cls_str(pred_H_test, y_test)}")


# ══════════════════════════════════════════════════════════════════════════════
# COMPARAISONS : Ridge polynomial, SVM RBF, Ridge RBF
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.svm import SVC
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV

print("\nCalibration Ridge polynomial...")
poly = PolynomialFeatures(degree=min(params["deg_P"], 8), include_bias=False)
ridge_poly = Ridge(alpha=1e-8)
ridge_poly.fit(poly.fit_transform(X_train), y_train)
pred_ridgepoly_te = ridge_poly.predict(poly.transform(X_test))

print("Calibration SVM RBF (GridSearchCV)...")
param_grid_svm = {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.01, 0.1, 1]}
svm = GridSearchCV(SVC(kernel="rbf"), param_grid_svm, cv=5, n_jobs=-1)
svm.fit(X_train, y_train)
pred_svm_te = svm.decision_function(X_test)

print("Calibration Ridge RBF (GridSearchCV)...")
param_grid_kr = {"alpha": [1e-3, 1e-2, 1e-1, 1.0], "gamma": [0.001, 0.01, 0.1, 1]}
kr = GridSearchCV(KernelRidge(kernel="rbf"), param_grid_kr, cv=5, n_jobs=-1)
kr.fit(X_train, y_train)
pred_kr_te = kr.predict(X_test)

print("\n" + "="*80)
print("RÉSUMÉ")
print("="*80)
print(f"Sobolev H (anisotrope, {sel.sum()}/{n_exp} experts) : {cls_str(pred_H_test, y_test)}")
print(f"Ridge polynomial                             : {cls_str(pred_ridgepoly_te, y_test)}")
print(f"SVM RBF     (best={svm.best_params_})          : {cls_str(pred_svm_te, y_test)}")
print(f"Ridge RBF   (best={kr.best_params_})          : {cls_str(pred_kr_te, y_test)}")
print(f"\nweights={params['weights']}  λ={params['lambda_reg']}  n_dirs={params['n_dirs']}")
print(f"n_train={params['n_train']}  n_unlabeled={params['n_unlabeled']}  n_test={params['n_test']}")
print(f"temps phase 1 : {sum(times_p1):.1f}s total ({np.mean(times_p1):.1f}s/expert)")
print("\nAccuracy/AUC individuelles des experts :")
for i in range(n_exp):
    tag = "" if sel[i] else "  (écarté)"
    print(f"  expert{i+1} : {cls_str(F_test[:,i], y_test)}  loss={expert_losses[i]:.3e}{tag}")

Génération du cloud sur {Q=0}...
  collectés:195/50 (cumulé 97.5%)
  collectés:19704/5000 (cumulé 98.5%)
  collectés:15576/3950 (cumulé 98.6%)
  X_train:(50, 5)  X_unlabeled:(3950, 5)
  cible binarisée au seuil médian=1.3608 : train (+1)=23/50  test (+1)=2554/5000

Phase 1 : 6 experts gaussiens anisotropes indépendants...
  expert 1/6 | rang=242/400 | loss=0.002938 (data=0.000078 reg=0.002860) | acc=0.8648 AUC=0.9459 | 15.5s
  expert 2/6 | rang=240/400 | loss=0.002934 (data=0.000086 reg=0.002848) | acc=0.8664 AUC=0.9467 | 14.8s
  expert 3/6 | rang=247/400 | loss=0.002932 (data=0.000089 reg=0.002844) | acc=0.8538 AUC=0.9398 | 14.7s
  expert 4/6 | rang=244/400 | loss=0.003005 (data=0.000086 reg=0.002919) | acc=0.8698 AUC=0.9477 | 14.5s
  expert 5/6 | rang=243/400 | loss=0.003035 (data=0.000087 reg=0.002948) | acc=0.8608 AUC=0.9425 | 14.5s
  expert 6/6 | rang=252/400 | loss=0.002782 (data=0.000076 reg=0.002705) | acc=0.8616 AUC=0.9459 | 14.5s

Phase 2 : Gram H entre 6 experts (n_G_H=3000)